In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track
import seaborn as sns

# Linalg libraries
import numpy as np
from scipy.stats import pearsonr

# Helper libraries
from dataclasses import dataclass


In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
sine_data = np.sin(np.pi * np.linspace(0, 5, 100))

In [ ]:
plt.plot(sine_data)

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length

    initial_state = [basis(2, 1)] + [basis(2, 0)] * (number_of_spins - 1)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]

    # Magnetic field terms to top row
    H = 0
    for i in [0, 1, 3]:        
        H -= driving_field * sz_list[i]

    
    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]

    return H

In [ ]:
# for strength in np.linspace(1, 10, 10, dtype=int):
simulation_parameters = SimulationParameters(
    length=5,
    coupling=1. * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

field_generator = 0
args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": field_generator
}

In [ ]:
observables = [qutip.rand_herm(
        2**1, 
        1.0, 
        dims=[[2] * 1, [2] * 1]
    )
    for _ in range(100)]

measure_sites = []
for _ in range(100):
    measure_sites.append(np.random.choice(5, 2, replace=False))

In [ ]:
state_dimension = 100

In [ ]:
res_measurements = []
measure_time = np.linspace(0, 0.05, 10)
state = simulation_state.quantum_state


for field_value in track(sine_data):
    args["driving"] = field_value
    results = mesolve(compute_hamiltonian, state, measure_time, [], [], args)
    state = results.states[-1]
    reservoir_state = []
    for i in range(state_dimension):
        outcome, state = qutip.measurement.measure(state, observables[i], [2])
        results = mesolve(compute_hamiltonian, state, measure_time, [], [], args)
        state = results.states[-1]
        reservoir_state.append(outcome)
    
    res_measurements.append(reservoir_state)


In [ ]:
np.shape(res_measurements)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()


        # self.readout_layer = nn.Sequential(
        #     nn.Linear(state_dimension, output_dimension),
        # )
        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, output_dimension)
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        # return self.readout_layer(x)
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        prediction_length: int,
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data).to(device)

        self.function_data = torch.Tensor(function_data).to(device)
        
        self.norm_factor = 1

    def split_data(self, train_size: float):
        """
        Split the data into training and validation sets.
        
        Parameters
        ----------
        train_size : float
                Size of the training set.
        """
        train_size = int(train_size * len(self))
        val_size = len(self) - train_size
        return torch.utils.data.random_split(self, [train_size, val_size])
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss = loss.item()
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn) -> np.ndarray:
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    
    return test_loss

In [ ]:
def train_model(dataset, test_ds, model = None):
    """
    Train a model on the current data.
    """    
    if model is None:
        model = NeuralNetwork(
            state_dimension=100,
            output_dimension=1
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Create the loader
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    
    # Train the network
    epochs = 2000
    train_losses = []
    test_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)
        loss = test(test_loader, model, loss_fn)
        test_losses.append(loss)

    train_losses = [item.cpu().detach().numpy() for item in train_losses]
    
    return train_losses, test_losses, model

## Train the model

In [ ]:
prediction_length = 1
train_fraction = 0.5

mix_train_indices = np.random.choice(len(res_measurements) - prediction_length - 1, int(train_fraction * len(res_measurements)), replace=False)
mix_test_indices = np.array([i for i in range(len(res_measurements) - prediction_length - 1) if i not in mix_train_indices])

sequence_train_indices = np.linspace(0, int(train_fraction * len(res_measurements)), int(train_fraction * len(res_measurements)), dtype=int)
sequence_test_indices = np.array([i for i in range(len(res_measurements) - prediction_length - 1) if i not in sequence_train_indices])

train_indices = sequence_train_indices
test_indices = sequence_test_indices

train_dataset = ReservoirDataset(
    state_data=np.array(res_measurements)[train_indices],
    function_data=sine_data[train_indices + prediction_length].reshape(-1, 1),
    prediction_length=1
)

test_dataset = ReservoirDataset(
    state_data=np.array(res_measurements)[test_indices],
    function_data=sine_data[test_indices + prediction_length].reshape(-1, 1),
    prediction_length=1
)
train_losses, test_losses, model = train_model(train_dataset, test_dataset, model=None)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_losses)
ax[1].plot(train_losses)

ax[1].set_yscale("log")
ax[0].set_yscale("log")

plt.show()

In [ ]:
test_predictions = []
test_targets = []
for item in test_dataset:
    state, target = item
    test_predictions.append(model(state).cpu().detach().numpy())
    test_targets.append(target.cpu().detach().numpy())

test_predictions = np.array(test_predictions)
test_targets = np.array(test_targets)

train_predictions = []
train_targets = []
for item in train_dataset:
    state, target = item
    train_predictions.append(model(state).cpu().detach().numpy())
    train_targets.append(target.cpu().detach().numpy())

train_predictions = np.array(train_predictions)
train_targets = np.array(train_targets)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_targets)
ax[0].plot(test_predictions)

ax[1].plot(train_targets)
ax[1].plot(train_predictions)